# AIRank — Exploratory Data Analysis
Complete EDA and model evaluation notebook.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils.data_loader import load_data, TASK_COLUMNS, FEATURE_COLUMNS
from utils.ml_model import train_model, predict_scores

df = load_data()
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
# Basic statistics
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nFeature statistics:')
df[FEATURE_COLUMNS].describe().round(2)

## 2. Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df[FEATURE_COLUMNS + ['coding', 'essay_writing', 'math_solving']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. AI Tool Performance Heatmap Across All Tasks

In [ ]:
task_matrix = df.set_index('tool_name')[TASK_COLUMNS]
fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(task_matrix, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Suitability Score'})
ax.set_title('AI Tool Performance Heatmap — All Task Categories',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Task Category')
ax.set_ylabel('AI Tool')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Linear Regression Model — Coding Task Example

In [ ]:
from sklearn.metrics import r2_score

metrics = train_model('coding')
print('=== Model: Coding Task ===')
print(f"R²   : {metrics['r2']:.4f}")
print(f"MAE  : {metrics['mae']:.4f}")
print(f"RMSE : {metrics['rmse']:.4f}")
print(f"CV Mean R²: {metrics['cv_mean']:.4f}")

## 5. Predicted vs Actual Scores — Scatter Plot

In [ ]:
result = predict_scores('coding')
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(result['actual_score'], result['predicted_score'],
           c='#534AB7', alpha=0.75, s=80, zorder=2)
for _, row in result.iterrows():
    ax.annotate(row['tool_name'], (row['actual_score'], row['predicted_score']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')
mn = min(result[['actual_score','predicted_score']].min())
mx = max(result[['actual_score','predicted_score']].max())
ax.plot([mn,mx], [mn,mx], 'r--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual Score')
ax.set_ylabel('Predicted Score')
ax.set_title('Predicted vs Actual — Coding Task', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. R² Scores Across All 25 Tasks

In [ ]:
all_r2 = {}
for task in TASK_COLUMNS:
    m = train_model(task)
    all_r2[task] = m['r2']

r2_series = pd.Series(all_r2).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#534AB7' if v >= 0.80 else '#D85A30' for v in r2_series]
ax.bar(range(len(r2_series)), r2_series.values, color=colors)
ax.axhline(0.80, color='#1D9E75', linestyle='--', label='Target R² = 0.80')
ax.set_xticks(range(len(r2_series)))
ax.set_xticklabels([t.replace('_',' ') for t in r2_series.index],
                    rotation=45, ha='right', fontsize=8)
ax.set_ylabel('R² Score')
ax.set_title('Model R² — All 25 Task Categories', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMean R²: {r2_series.mean():.4f}")
print(f"Tasks meeting target: {(r2_series >= 0.80).sum()}/{len(r2_series)}")